In [0]:
print("Clinical Trial Data Pipeline")
print("Databricks environment is working!")

Clinical Trial Data Pipeline
Databricks environment is working!


In [0]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [0]:
import requests

url = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "pageSize": 100,
    "format": "json"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print("Successfully connected to ClinicalTrials.gov!")
print("Number of studies retrieved:", len(data["studies"]))

Successfully connected to ClinicalTrials.gov!
Number of studies retrieved: 100


In [0]:
import requests
import json

url = "https://clinicaltrials.gov/api/v2/studies"

all_studies = []
next_page_token = None

for page in range(10):

    params = {
        "pageSize": 100,
        "format": "json"
    }

    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    studies = data.get("studies", [])
    all_studies.extend(studies)

    print(f"Batch {page + 1}: Retrieved {len(studies)} studies")

    next_page_token = data.get("nextPageToken")

    if not next_page_token:
        break

print(f"\nTotal clinical trials retrieved: {len(all_studies)}")

Batch 1: Retrieved 100 studies
Batch 2: Retrieved 100 studies
Batch 3: Retrieved 100 studies
Batch 4: Retrieved 100 studies
Batch 5: Retrieved 100 studies
Batch 6: Retrieved 100 studies
Batch 7: Retrieved 100 studies
Batch 8: Retrieved 100 studies
Batch 9: Retrieved 100 studies
Batch 10: Retrieved 100 studies

Total clinical trials retrieved: 1000


In [0]:
from pyspark.sql import Row
from datetime import datetime, timezone

ingestion_time = datetime.now(timezone.utc).isoformat()

records = [
    Row(
        nct_id=study.get("protocolSection", {})
                    .get("identificationModule", {})
                    .get("nctId"),
        raw_json=json.dumps(study),
        ingested_at=ingestion_time
    )
    for study in all_studies
]

bronze_df = spark.createDataFrame(records)

display(bronze_df.limit(5))

nct_id raw_json ingested_at NCT02976935 {"protocolSection": {"identificationModule": {"nctId": "NCT02976935", "orgStudyIdInfo": {"id": "13104"}, "organization": {"fullName": "University of Nottingham", "class": "OTHER"}, "briefTitle": "Functional MR Lung Imaging Using Hyperpolarised 129Xe", "officialTitle": "FUNCTIONAL MAGNETIC RESONANCE LUNG IMAGING USING INHALED HYPERPOLARISED 129Xe"}, "statusModule": {"statusVerifiedDate": "2022-12", "overallStatus": "COMPLETED", "expandedAccessInfo": {"hasExpandedAccess": false}, "startDateStruct": {"date": "2014-02"}, "primaryCompletionDateStruct": {"date": "2021-12-31", "type": "ACTUAL"}, "completionDateStruct": {"date": "2021-12-31", "type": "ACTUAL"}, "studyFirstSubmitDate": "2016-02-09", "studyFirstSubmitQcDate": "2016-11-24", "studyFirstPostDateStruct": {"date": "2016-11-30", "type": "ESTIMATED"}, "lastUpdateSubmitDate": "2022-12-21", "lastUpdatePostDateStruct": {"date": "2022-12-22", "type": "ACTUAL"}}, "sponsorCollaboratorsModule": {"responsibleParty": {"type": "SPONSOR"}, "leadSponsor": {"name": "University of Nottingham", "class": "OTHER"}}, "oversightModule": {"oversightHasDmc": false}, "descriptionModule": {"briefSummary": "The aim of this study is to investigate the potential value of a novel imaging technique (hyperpolarized 129Xe lung imaging) in the diagnosis and assessment of lung disease in patients with COPD and IPF."}, "conditionsModule": {"conditions": ["COPD", "IPF"]}, "designModule": {"studyType": "OBSERVATIONAL", "patientRegistry": false, "designInfo": {"observationalModel": "COHORT", "timePerspective": "PROSPECTIVE"}, "enrollmentInfo": {"count": 53, "type": "ACTUAL"}}, "armsInterventionsModule": {"armGroups": [{"label": "Healthy Volunteers", "description": "Subjects will receive 1L non-hyperpolarised 129Xenon to inhale. Then:Subjects will inhale the first hyperpolarised 129Xenon calibration dose (up to 1L, which may be a mix of hyperpolarised 129Xenon and N2) and hold their breath while the MRI scan is performed. Then:A further series of inhalations will be performed up to a maximum of 3 (maximum total exposure to hyperpolarised 129Xenon will be 3.6L). Optional Study Visit #2:Identical to Study Visit 1 but with the further series of inhalations being up to a maximum of 4 (maximum total exposure to hyperpolarised 129Xenon will be 5L).Optional Study Visit #3: Identical to Study Visit 2 (maximum total exposure to hyperpolarised 129Xenon will be 5L)", "interventionNames": ["Other: hyperpolarised 129 Xenon"]}, {"label": "Chronic Obstructive Pulmonary Disease", "description": "Subjects will receive 1L bag of non-hyperpolarised 129Xenon to inhale. Then:Subjects will inhale the first hyperpolarised 129Xenon calibration dose (up to 1L, which may be a mix of hyperpolarised 129Xenon and N2) and hold their breath for required time while the MRI scan is performed.Then:A further series of inhalations will be performed up to a maximum of 3 (maximum total exposure to hyperpolarised 129Xenon will be 3.6L). Optional Study Visit #2: Identical to Study Visit 1 (maximum total exposure to hyperpolarised 129Xenon will be 3.6L)", "interventionNames": ["Other: hyperpolarised 129 Xenon"]}], "interventions": [{"type": "OTHER", "name": "hyperpolarised 129 Xenon", "description": "Hyperpolarised 129 Xe to assess lung function", "armGroupLabels": ["Chronic Obstructive Pulmonary Disease", "Healthy Volunteers"]}]}, "outcomesModule": {"primaryOutcomes": [{"measure": "Acquisition of baseline data and images to investigate use of hp129Xe as a diagnostic tool", "description": "i) Obtain baseline data for each group with matched clinical data (e.g. pulmonary function tests) allowing comparison with existing clinical diagnostic techniques and also determining correlation with disease severity ii) Use the images obtained to set up standard sequencing protocols and define algorithms and inform power calculations for subsequent studies", "timeFrame": "4 years"}], "secondaryOutcomes": [{"measure": "Collatio

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("clinical_trials_bronze")

print("Bronze Delta table created successfully!")

Bronze Delta table created successfully!


In [0]:

%sql

SELECT
    COUNT(*) AS total_studies,
    COUNT(DISTINCT nct_id) AS unique_studies
FROM clinical_trials_bronze;

total_studies,unique_studies
1000,1000


In [0]:
%sql
SELECT
    nct_id,
    LENGTH(raw_json) AS json_length,
    ingested_at
FROM clinical_trials_bronze
LIMIT 5;


nct_id,json_length,ingested_at
NCT02976935,7265,2026-10-29T17:26:32.235207+00:00
NCT05877235,7776,2026-10-29T17:26:32.235207+00:00
NCT06273735,15246,2026-10-29T17:26:32.235207+00:00
NCT07582835,24892,2026-10-29T17:26:32.235207+00:00
NCT01831635,43955,2026-10-29T17:26:32.235207+00:00


In [0]:
from pyspark.sql.functions import (
    col,
    get_json_object,
    to_date,
    regexp_replace
)

# Read our Bronze Delta table
bronze_df = spark.table("clinical_trials_bronze")

# Extract structured fields from the raw JSON
silver_df = bronze_df.select(
    col("nct_id"),

    get_json_object(
        col("raw_json"),
        "$.protocolSection.identificationModule.briefTitle"
    ).alias("study_title"),

    get_json_object(
        col("raw_json"),
        "$.protocolSection.statusModule.overallStatus"
    ).alias("recruitment_status"),

    get_json_object(
        col("raw_json"),
        "$.protocolSection.designModule.phases[0]"
    ).alias("study_phase"),

    get_json_object(
        col("raw_json"),
        "$.protocolSection.designModule.enrollmentInfo.count"
    ).cast("int").alias("enrollment"),

    get_json_object(
        col("raw_json"),
        "$.protocolSection.conditionsModule.conditions[0]"
    ).alias("medical_condition"),

    get_json_object(
        col("raw_json"),
        "$.protocolSection.statusModule.startDateStruct.date"
    ).alias("start_date_raw")
)

# Convert start dates into a standardized date format
silver_df = silver_df.withColumn(
    "start_date",
    to_date(
        regexp_replace(col("start_date_raw"), r"^(\d{4}-\d{2})$", "$1-01"),
        "yyyy-MM-dd"
    )
).drop("start_date_raw")

display(silver_df.limit(10))

nct_id,study_title,recruitment_status,study_phase,enrollment,medical_condition,start_date
NCT02976935,Functional MR Lung Imaging Using Hyperpolarised 129Xe,COMPLETED,null,53,COPD,2014-02-01
NCT05877235,Serum Concentration of Endogenous Estrogens and Sirtuin-1 After Administration of Atorvastatin and Quercetin:,COMPLETED,NA,60,Coronary Artery Disease,2021-10-03
NCT06273735,Prospective Trial: Acute Myocardial Injury After Radical Prostatectomy,COMPLETED,null,1000,Myocardial Injury,2023-08-14
NCT07582835,Study of the Long-term Effects of P2Y12 Inhibitor Monotherapy and Coagulation Markers After Percutaneous Coronary Angioplasty.,RECRUITING,PHASE3,355,Coronary Artery Disease,2026-04-20
NCT01831635,Myeloproliferative Neoplasms: an In-depth Case-control Study,COMPLETED,null,234,Polycythemia Vera,2013-04-01
NCT04148235,A Pilot Study to Explore the Role of Gut Flora in Celiac Disease,WITHDRAWN,null,0,Celiac Disease,2020-03-02
NCT04728035,Study of Irinotecan Liposome Injection in Patients With Advanced Breast Cance,COMPLETED,PHASE1,119,Advanced Breast Cancer,2021-05-31
NCT02056535,Screening and Brief Intervention in the ED Among Mexican-origin Young Adults,COMPLETED,PHASE2,698,Problem Drinking,2010-05-01
NCT03524235,"Haploidentical Stem Cell Transplant with Prophylactic Natural Killer DLI for Lymphoma, Multiple Myeloma, and CLL",COMPLETED,PHASE1,20,Multiple Myeloma,2018-07-18
NCT03417635,"Meditation Strategies, Attention, and Mobility in Older Adults",COMPLETED,NA,43,Mobility Limitation,2019-01-02


In [0]:
from pyspark.sql.functions import count, when

silver_df.select([
    count(when(col(c).isNull(), c)).alias(c + "_missing")
    for c in silver_df.columns
]).show()

+--------------+-------------------+--------------------------+-------------------+------------------+-------------------------+------------------+
|nct_id_missing|study_title_missing|recruitment_status_missing|study_phase_missing|enrollment_missing|medical_condition_missing|start_date_missing|
+--------------+-------------------+--------------------------+-------------------+------------------+-------------------------+------------------+
|             0|                  0|                         0|                241|                13|                        2|                 7|
+--------------+-------------------+--------------------------+-------------------+------------------+-------------------------+------------------+



In [0]:
silver_df = silver_df.dropDuplicates(["nct_id"])

print("Total records after removing duplicates:", silver_df.count())

Total records after removing duplicates: 1000


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("clinical_trials_silver")

print("Silver Delta table created successfully!")

Silver Delta table created successfully!


In [0]:
%sql
SELECT
    COUNT(*) AS total_studies,
    COUNT(DISTINCT nct_id) AS unique_studies,
    COUNT(study_title) AS studies_with_titles,
    COUNT(enrollment) AS studies_with_enrollment
FROM clinical_trials_silver;


total_studies,unique_studies,studies_with_titles,studies_with_enrollment
1000,1000,1000,987


In [0]:
%sql

CREATE OR REPLACE TABLE clinical_trials_gold_status
USING DELTA
AS
SELECT
    COALESCE(recruitment_status, 'UNKNOWN') AS recruitment_status,
    COUNT(*) AS total_trials,
    SUM(COALESCE(enrollment, 0)) AS total_reported_enrollment,
    ROUND(AVG(enrollment), 1) AS average_enrollment
FROM clinical_trials_silver
GROUP BY COALESCE(recruitment_status, 'UNKNOWN');

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM clinical_trials_gold_status
ORDER BY total_trials DESC;

recruitment_status,total_trials,total_reported_enrollment,average_enrollment
COMPLETED,557,243766,444.0
UNKNOWN,153,222326,1462.7
RECRUITING,118,95603,810.2
TERMINATED,58,15026,263.6
NOT_YET_RECRUITING,39,12142,311.3
ACTIVE_NOT_RECRUITING,36,160610,4461.4
WITHDRAWN,32,0,0.0
ENROLLING_BY_INVITATION,3,30301,10100.3
WITHHELD,2,0,null
SUSPENDED,1,54,54.0


In [0]:
%sql

CREATE OR REPLACE TABLE clinical_trials_gold_phase
USING DELTA
AS
SELECT
    COALESCE(study_phase, 'NOT_APPLICABLE') AS study_phase,
    COUNT(*) AS total_trials,
    ROUND(AVG(enrollment), 1) AS average_enrollment
FROM clinical_trials_silver
GROUP BY COALESCE(study_phase, 'NOT_APPLICABLE');


num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM clinical_trials_gold_phase
ORDER BY total_trials DESC;

study_phase,total_trials,average_enrollment
NA,390,476.5
NOT_APPLICABLE,241,2243.3
PHASE2,121,125.8
PHASE1,115,53.1
PHASE3,67,592.6
PHASE4,57,183.9
EARLY_PHASE1,9,31.4


In [0]:
%sql

CREATE OR REPLACE TABLE clinical_trials_gold_condition
USING DELTA
AS
SELECT
    COALESCE(medical_condition, 'NOT_SPECIFIED') AS medical_condition,
    COUNT(*) AS total_trials,
    ROUND(AVG(enrollment), 1) AS average_enrollment
FROM clinical_trials_silver
GROUP BY COALESCE(medical_condition, 'NOT_SPECIFIED');


num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM clinical_trials_gold_condition
ORDER BY total_trials DESC
LIMIT 15;


medical_condition,total_trials,average_enrollment
Healthy,27,112.3
Breast Cancer,12,711.7
Hypertension,9,860.8
Obesity,9,146.0
Type 2 Diabetes,8,124.9
Stroke,7,130.9
Asthma,7,99.3
Coronary Artery Disease,7,1744.6
Heart Failure,7,576.0
HIV,6,2337.8
